# FNAN 632 - Project Part 2

For this part of the project I used daily market data for the S&P 500, IBM, and Coca-Cola. I calculated daily returns, summary statistics, and then compared the individual stocks with an equal-weight portfolio.


## Setup


In [ ]:
# If yfinance is not installed, run this line once:
# !pip install yfinance

import yfinance as yf
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt


## 1. Download S&P 500 Daily Prices

I downloaded a little extra data at the end of 1969 because the first 1970 return needs the previous trading day's price.


In [ ]:
sp500_data = yf.download("^GSPC", start="1969-12-01", end="2026-01-01",
                           auto_adjust=False, progress=False)

# yfinance can return a MultiIndex, so this keeps the adjusted close as a Series
if isinstance(sp500_data.columns, pd.MultiIndex):
    sp500_price = sp500_data["Adj Close"].iloc[:, 0]
else:
    sp500_price = sp500_data["Adj Close"]

sp500_price.head()


## 2. Calculate Daily Returns


In [ ]:
# Calculate returns first so the first trading day in 1970 can use the last 1969 price
sp500_all_returns = sp500_price.pct_change()

sp500_returns = sp500_all_returns.loc[
    (sp500_all_returns.index >= "1970-01-01") &
    (sp500_all_returns.index < "2026-01-01")
].dropna()

print("Sample size:", sp500_returns.count())
sp500_returns.head()


The sample size above is the number of daily S&P 500 returns from 1970 through the end of 2025.


## 3. S&P 500 Plots


### 3(a). Historical Price Trajectory


In [ ]:
sp500_plot_price = sp500_price.loc[
    (sp500_price.index >= "1970-01-01") &
    (sp500_price.index < "2026-01-01")
]

plt.figure(figsize=(10, 5))
plt.plot(sp500_plot_price.index, sp500_plot_price.values)
plt.title("S&P 500 Historical Price, 1970-2025")
plt.xlabel("Date")
plt.ylabel("Adjusted Close")
plt.grid(alpha=0.25)
plt.show()


### 3(b). Histogram of Daily Returns


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(sp500_returns, bins=50, edgecolor="black")
plt.title("Histogram of S&P 500 Daily Returns")
plt.xlabel("Daily Return")
plt.ylabel("Frequency")
plt.show()


## 4. Mean, Standard Deviation, Skewness, and Excess Kurtosis


In [ ]:
mu = sp500_returns.mean()
sigma = sp500_returns.std()
sp_skew = sp500_returns.skew()
sp_kurt = sp500_returns.kurt()  # pandas reports excess kurtosis

print("Arithmetic mean return:", mu)
print("Standard deviation:", sigma)
print("Skewness:", sp_skew)
print("Excess kurtosis:", sp_kurt)


The mean gives the average daily return, while the standard deviation shows the typical amount of daily variation. Skewness shows whether the distribution is more stretched on one side, and excess kurtosis shows how heavy the tails are compared with a normal distribution.


## 5. Maximum Daily Return


In [ ]:
max_return = sp500_returns.max()
max_date = sp500_returns.idxmax()
max_z = (max_return - mu) / sigma

print("Maximum daily return:", max_return)
print("Date:", max_date.date())
print("Standard deviations above the mean:", max_z)


I measured the maximum return in standard deviations using its distance from the sample mean divided by the sample standard deviation.


## 6. Minimum Daily Return


In [ ]:
min_return = sp500_returns.min()
min_date = sp500_returns.idxmin()
min_z = (min_return - mu) / sigma

print("Minimum daily return:", min_return)
print("Date:", min_date.date())
print("Standard deviations from the mean:", min_z)


## 7. Fit a Normal Distribution to the S&P 500 Returns


In [ ]:
m = sp500_returns.min()
M = sp500_returns.max()

xvalues = np.arange(m, M, .001)
yvalues = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(
    -(xvalues - mu)**2 / (2 * sigma**2)
)

plt.figure(figsize=(9, 5))
plt.hist(sp500_returns, bins=50, density=True, alpha=0.6, edgecolor="black",
         label="S&P 500 returns")
plt.plot(xvalues, yvalues, linewidth=2, label="Fitted normal distribution")
plt.title("S&P 500 Returns vs. Fitted Normal Distribution")
plt.xlabel("Daily Return")
plt.ylabel("Density")
plt.legend()
plt.show()


The normal curve uses the same mean and standard deviation as the actual S&P 500 return sample. This makes it easier to compare the shape of the real return distribution with a normal distribution.


## 8. Quartiles: Actual Returns vs. Normal Distribution


In [ ]:
actual_q1 = sp500_returns.quantile(0.25)
actual_q2 = sp500_returns.quantile(0.50)
actual_q3 = sp500_returns.quantile(0.75)

normal_q1 = mu - 0.675 * sigma
normal_q3 = mu + 0.675 * sigma

quartile_table = pd.DataFrame({
    "Actual S&P 500": [actual_q1, actual_q2, actual_q3],
    "Normal Distribution": [normal_q1, mu, normal_q3]
}, index=["Q1", "Q2 / Median", "Q3"])

quartile_table


The actual quartiles do not have to match the fitted normal distribution exactly. If they are different, it means the center and spread of the real returns are not perfectly normal. The skewness and excess kurtosis from Question 4 also help explain why the actual return distribution can have a different shape, especially in the tails.


## 9. Return Intervals, Frequencies, and Relative Frequencies


In [ ]:
rows = []

# Lower tail: returns less than or equal to mu - k*sigma
for k in range(10, 0, -1):
    cutoff = mu - k * sigma
    count = (sp500_returns <= cutoff).sum()
    rows.append([f"returns <= mu - {k} sigma", cutoff, count, count / sp500_returns.count()])

# Upper tail: returns greater than or equal to mu + k*sigma
for k in range(1, 11):
    cutoff = mu + k * sigma
    count = (sp500_returns >= cutoff).sum()
    rows.append([f"returns >= mu + {k} sigma", cutoff, count, count / sp500_returns.count()])

frequency_table = pd.DataFrame(
    rows,
    columns=["Return Interval", "Raw Cutoff", "Frequency", "Relative Frequency"]
)

frequency_table


## 10. How Likely is a Return Beyond 5 Standard Deviations?


In [ ]:
beyond_5 = ((sp500_returns <= mu - 5*sigma) |
            (sp500_returns >= mu + 5*sigma))

beyond_5_count = beyond_5.sum()
beyond_5_frequency = beyond_5.mean()

print("Number of observations beyond +/- 5 sigma:", beyond_5_count)
print("Historical relative frequency:", beyond_5_frequency)
print("Normal distribution 5-sigma probability given in assignment:", 3e-7)
print("Historical frequency / normal probability:", beyond_5_frequency / 3e-7)


Based on the historical sample, 5-standard-deviation moves happen more often than a normal distribution would suggest if the historical relative frequency above is larger than 0.0000003. This is evidence that stock market returns have heavier tails than the normal distribution.


## 11. Download IBM and Coca-Cola Data and Calculate Daily Returns


In [ ]:
def get_adjusted_close(ticker):
    data = yf.download(ticker, start="1969-12-01", end="2026-01-01",
                       auto_adjust=False, progress=False)
    if isinstance(data.columns, pd.MultiIndex):
        return data["Adj Close"].iloc[:, 0]
    return data["Adj Close"]

ibm_price = get_adjusted_close("IBM")
ko_price = get_adjusted_close("KO")

ibm_returns = ibm_price.pct_change().loc["1970-01-01":"2025-12-31"].dropna()
ko_returns = ko_price.pct_change().loc["1970-01-01":"2025-12-31"].dropna()

print("IBM observations:", ibm_returns.count())
print("KO observations:", ko_returns.count())


## 12. IBM and Coca-Cola Summary Statistics


In [ ]:
def stats(series):
    return pd.Series({
        "Mean": series.mean(),
        "Standard Deviation": series.std(),
        "Skewness": series.skew(),
        "Excess Kurtosis": series.kurt()
    })

comparison = pd.DataFrame({
    "S&P 500": stats(sp500_returns),
    "IBM": stats(ibm_returns),
    "Coca-Cola (KO)": stats(ko_returns)
}).T

comparison


This table makes the three return distributions easier to compare. I would expect the individual stocks to have more company-specific movement than the broad S&P 500 index, which can show up in their standard deviations and tail statistics.


## 13. Correlations


In [ ]:
combined = pd.concat(
    [sp500_returns.rename("S&P 500"),
     ibm_returns.rename("IBM"),
     ko_returns.rename("KO")],
    axis=1, join="inner"
).dropna()

corr_ibm_sp = combined["IBM"].corr(combined["S&P 500"])
corr_ko_sp = combined["KO"].corr(combined["S&P 500"])
corr_ibm_ko = combined["IBM"].corr(combined["KO"])

print("(a) IBM and S&P 500:", corr_ibm_sp)
print("(b) KO and S&P 500:", corr_ko_sp)
print("(c) IBM and KO:", corr_ibm_ko)


A positive correlation means the two returns tend to move in the same direction. A correlation closer to 1 means the relationship is stronger, while a value closer to 0 means the linear relationship is weaker.


## 14. Equal-Weight IBM and Coca-Cola Portfolio


In [ ]:
stock_returns = pd.concat(
    [ibm_returns.rename("IBM"), ko_returns.rename("KO")],
    axis=1, join="inner"
).dropna()

# Equal value weights, rebalanced daily
portfolio_returns = 0.5 * stock_returns["IBM"] + 0.5 * stock_returns["KO"]
portfolio_returns.name = "Portfolio"

portfolio_returns.head()


Since the portfolio is rebalanced daily to equal weights, each day the portfolio return is 50% of the IBM return plus 50% of the Coca-Cola return.


## 15. Portfolio Summary Statistics


In [ ]:
portfolio_stats = stats(portfolio_returns)
portfolio_stats


## 16. Correlation of Each Stock with the Portfolio


In [ ]:
corr_ibm_port = stock_returns["IBM"].corr(portfolio_returns)
corr_ko_port = stock_returns["KO"].corr(portfolio_returns)

print("(a) IBM and portfolio:", corr_ibm_port)
print("(b) KO and portfolio:", corr_ko_port)


## 17. Portfolio Mean and Standard Deviation Compared with Weighted Averages


In [ ]:
weighted_mean = 0.5 * ibm_returns.mean() + 0.5 * ko_returns.mean()
weighted_std = 0.5 * ibm_returns.std() + 0.5 * ko_returns.std()

print("Portfolio mean:", portfolio_returns.mean())
print("50/50 weighted average of stock means:", weighted_mean)
print()
print("Portfolio standard deviation:", portfolio_returns.std())
print("50/50 weighted average of stock standard deviations:", weighted_std)


The portfolio mean return should equal the 50/50 weighted average of the two stock mean returns, apart from tiny differences caused by the dates used in the samples. The portfolio standard deviation is generally lower than the weighted average of the two individual standard deviations because IBM and Coca-Cola are not perfectly correlated. Their movements partly offset each other, which reduces the portfolio risk.


## 18. Portfolio vs. Concentrating in One Stock


Based on these results, holding the portfolio gives a diversification benefit. The portfolio still combines the returns from IBM and Coca-Cola, but its standard deviation should be lower than simply taking the weighted average of their individual standard deviations. This happens because the two stocks do not move exactly together every day. Concentrating everything in one stock leaves the investor more exposed to company-specific risk, while combining stocks can reduce some of that risk.


## Final Check


In [ ]:
print("S&P 500 sample:", sp500_returns.index.min().date(), "to", sp500_returns.index.max().date())
print("S&P 500 observations:", len(sp500_returns))
print("IBM observations:", len(ibm_returns))
print("KO observations:", len(ko_returns))
print("Portfolio observations:", len(portfolio_returns))
